**FIRST WE IMPORT THE REQUIRED LIBRARIES FRO THE SAKE OF THE CODE**

In [1]:
import json
import random

**NOW WE HAVE TO UPLOAD THE "graph_with_metadata.json" FILE INTO OUR CODE**

In [2]:
def load_graph(a = "graph_with_metadata.json"):

  with open(a, "r") as b:
    data = json.load(b)

  adj = data["adjacency"]
  adj_ = {}
  for k , l in adj.items():
    A = []
    for m , n in l:
      A.append((m,n))
    adj_[int(k)] = A


  pos = data["positions"]
  pos_ = {}
  for k , l in pos.items():
   pos_[int(k)] = list(l)

  metadata = data.get("metadata", {})
  x = metadata.get("exit_nodes", [])
  exit_nodes = x[1]

  print("Graph was successfully loaded in the code")

  return adj_, pos_, metadata, exit_nodes

**NOW WE DEFINE THE DIJKSTRA ALGORITHM FROM SCRATCH**

In [5]:
def Dijkstra(adj_ , s_node , e_node):
    
    nodes = list(adj_.keys())
    n = len(nodes)


    dist = [100000] * n
    prev = [None] * n
    dist[s_node] = 0

    K = set()
    V = set(list(range(n)))
    Q = V.difference(K)


    while Q:
        mindist = 100000
        u = None

        for node in Q:
            if dist[node] < mindist:
                mindist = dist[node]
                u = node


        if u is None or dist[u] == 100000:
            break


        for v, weight in adj_.get(u, []):
            if v in Q:
                alt = dist[u] + weight
                if alt < dist[v]:
                    prev[v] = u
                    dist[v] = alt


        K.add(u)
        Q = V.difference(K)


        if u == e_node:
            break


    if dist[e_node] < 100000:
        target = e_node
    else:
        reachable = [i for i in range(n) if dist[i] < 100000]
        reachable_except_source = [i for i in reachable if i != s_node]

        if reachable_except_source:
            target = min(reachable_except_source, key=lambda i: dist[i])

           # if the target node is not reachable then the dijkstra will direct the car to the next nearest node
        elif reachable:
            target = s_node
        else:
            return dist, [s_node]


    path = []
    x = target
    while x is not None:
        path.insert(0, x)
        x = prev[x]

    return dist, path


**NOW WE INITIALIZE DIFFERENT KINDS OF EVENTS AND THEIR EFFECTS ON THE EDGE WEIGHTS**

In [6]:
def generate_random_events(adj_, step):
    events = []

    for node in adj_:
        for n, w in adj_[node]:

            if random.random() <= 0.3:  #probability of any event happening is 0.3
                event_type = random.choice(['traffic', 'blockage', 'oneway'])
                #In that 0.3, these three events have equal probability of occuring i.e, each
                #event has a probability of 0.1

                if event_type == 'traffic':
                    events.append({
                        'type': 'traffic',
                        'edge': (node, n),
                        'original_weight': w,
                        'multiplier': random.uniform(2, 2.5),
                        'expires_at': step + 3})

                elif event_type == 'blockage':
                    events.append({
                        'type': 'blockage',
                        'edge': (node, n),
                        'original_weight': w,
                        'expires_at': step + 4})


                else:
                    reverse_exists = False
                    reverse_weight = None
                    if n in adj_:
                        for a, w in adj_[n]:
                            if a == node:
                                reverse_exists = True
                                reverse_weight = w
                                break

                    if reverse_exists:
                        events.append({
                            'type': 'oneway',
                            'edge': (node, n),
                            'reverse_edge': (n, node),
                            'reverse_weight': reverse_weight,
                            'expires_at': step + 6

                        })

    return events
# In the third case where the event_type is 'oneway'. I wasn't sure what do if the edge didnt have a
# reverse edge. So I didn't apply any changes to that edge if its event_type was 'oneway' and didn't have
# a reverse edge.

**APPLYING THE ABOVE CREATED EVENTS INTO OUR GRAPH**


**AND ALSO REVERTING BACK THE CHANGES MADE AFTER CERTAIN TIME**

In [7]:
def apply_event(adj_, event):


    if event['type'] == 'traffic':
      f_node, t_node = event['edge']
      new_w = event['original_weight'] * event['multiplier']
      for i, (n, w) in enumerate(adj_[f_node]):
        if n == t_node:
            adj_[f_node][i] = (n, new_w)
            break


    elif event['type'] == 'blockage':
        f_node, t_node = event['edge']
        for i, (n, w) in enumerate(adj_[f_node]):
          if n == t_node:
            adj_[f_node][i] = (n, 100000)
            # I wanted to preserve the index of the corresponding (n,w). So instead of deleting the edge, i'm simply increasing
            # the weight to 100000 so that during dijkstra this edge will simply not be selected as teh weight is too high
            break


    elif event['type'] == 'oneway':
        f_node, t_node = event['reverse_edge']
        for i, (n, w) in enumerate(adj_[f_node]):
          if n == t_node:
            adj_[f_node][i] = (n, 100000)
            # Similar to above case, here i simply increased the weight of the reverse edge to 100000
            break

In [8]:
def revert_event(adj_, event):

    if event['type'] == 'traffic':
        f_node, t_node = event['edge']
        for i, (n, w) in enumerate(adj_[f_node]):
          if n == t_node:
            adj_[f_node][i] = (n, event['original_weight'])
            break

    elif event['type'] == 'blockage':
        f_node, t_node = event['edge']
        for i, (n, w) in enumerate(adj_[f_node]):
          if n == t_node:
            adj_[f_node][i] = (n,event['original_weight'])
            break

    elif event['type'] == 'oneway':
        f_node, t_node = event['reverse_edge']
        for i, (n, w) in enumerate(adj_[f_node]):
          if n == t_node:
            adj_[f_node][i] = (n,event['reverse_weight'])
            break

**NOW LETS DEFINE A FUNCTION TO CHECK WHETHER CAR-A HAS BEEN CAUGHT BY CAR-B OR NOT**

In [9]:
def caught_or_not(carA, carB):

    # if both the cars are on the same node
    if carA['pos'] == carB['pos'] and carA.get('edge_from') is None and carB.get('edge_from') is None:
        return True

    # if both the cars are on the same edge within 0.5 distance
    if carA.get('edge_from') == carB.get('edge_from') and carA.get('edge_to') == carB.get('edge_to') and carA.get('edge_from') is not None:
        if abs(carA['progress'] - carB['progress']) < 0.5:
            return True

    return False

**BRINGING ALL THE PREDEFINED FUNCTIONS INTO A SINGLE "chase" FUNCTION WHICH WILL SIMULATE THE ENTIRE CHASE SEQUENCE**

In [10]:
def chase(adj_, exit_nodes, m_steps):

    carA = {
        'pos': 0,
        'speed': 1.0,
        'path': None,
        'path_index': 0,
        'edge_from': None,
        'edge_to': None,
        'progress': 0
    }

    carB = {
        'pos': 49,
        'speed': 1.5,
        'path': None,
        'path_index': 0,
        'edge_from': None,
        'edge_to': None,
        'progress': 0,
        'delay': 3
    }

    # Computing the initial path for Car A
    _, path = Dijkstra(adj_, carA['pos'], exit_nodes)
    carA['path'] = path
    carA['path_index'] = 0

    simulation_log = []
    active_events = []
    caught = False
    reached = False

    for step in range(m_steps):
        log_events = []

        # Moving Car A
        if not reached and carA['path'] and len(carA['path']) > 1:
            if carA['edge_from'] is None:
                # Start moving on next edge
                current_idx = carA['path_index']
                if current_idx < len(carA['path']) - 1:
                    carA['edge_from'] = carA['path'][current_idx]
                    carA['edge_to'] = carA['path'][current_idx + 1]
                    carA['progress'] = 0

                    # Get edge weight
                    edge_weight = None
                    if carA['edge_from'] in adj_:
                        for n, w in adj_[carA['edge_from']]:
                            if n == carA['edge_to']:
                                edge_weight = w
                                break

                    if edge_weight is None:
                        # Path no longer valid, need to recompute
                        _, path = Dijkstra(adj_, carA['pos'], exit_nodes)
                        carA['path'] = path
                        carA['path_index'] = 0
                        for i in range(len(path)):
                            if path[i] == carA['pos']:
                                carA['path_index'] = i
                                break
                        carA['edge_from'] = None
                    else:
                        carA['edge_weight'] = edge_weight

            if carA['edge_from'] is not None:
                # Continue moving on current edge
                carA['progress'] += carA['speed']
                if carA['progress'] >= carA['edge_weight']:
                    carA['pos'] = carA['edge_to']
                    carA['path_index'] += 1
                    carA['edge_from'] = None
                    carA['edge_to'] = None
                    carA['progress'] = 0


                    if carA['pos'] == exit_nodes:
                        reached = True

        # Move Car B
        if carB['delay'] > 0:
            carB['delay'] -= 1
            if carB['delay'] == 0:
                target_pos = carA['edge_to'] if carA['edge_to'] is not None else carA['pos']
                _, path = Dijkstra(adj_, carB['pos'], target_pos)
                carB['path'] = path
                carB['path_index'] = 0
        elif not caught:
            if carB['edge_from'] is None:
                current_idx = carB['path_index']
                if current_idx < len(carB['path']) - 1:
                    carB['edge_from'] = carB['path'][current_idx]
                    carB['edge_to'] = carB['path'][current_idx + 1]
                    carB['progress'] = 0


                    edge_weight = None
                    if carB['edge_from'] in adj_:
                        for n, w in adj_[carB['edge_from']]:
                            if n == carB['edge_to']:
                                edge_weight = w
                                break

                    if edge_weight is None:
                        # The path no longer valid, hence we have to recompute
                        target_pos = carA['edge_to'] if carA['edge_to'] is not None else carA['pos']
                        _, path = Dijkstra(adj_, carB['pos'], target_pos)
                        carB['path'] = path
                        carB['path_index'] = 0
                        for i in range(len(path)):
                            if path[i] == carB['pos']:
                                carB['path_index'] = i
                                break
                        carB['edge_from'] = None
                    else:
                        carB['edge_weight'] = edge_weight

            if carB['edge_from'] is not None:
                # Continue to move on current edge
                carB['progress'] += carB['speed']
                if carB['progress'] >= carB['edge_weight']:
                    carB['pos'] = carB['edge_to']
                    carB['path_index'] += 1
                    carB['edge_from'] = None
                    carB['edge_to'] = None
                    carB['progress'] = 0

                    # Recomputing its path to Car A's current position
                    target_pos = carA['edge_to'] if carA['edge_to'] is not None else carA['pos']
                    _, path = Dijkstra(adj_, carB['pos'], target_pos)
                    carB['path'] = path
                    carB['path_index'] = 0

        # Check catch condition
        caught = caught_or_not(carA, carB)
        if caught: reached = False


        new_events = generate_random_events(adj_, step)
        for new_event in new_events:
            active_events.append(new_event)
            apply_event(adj_, new_event)

            event_desc = f"{new_event['type'].capitalize()}"
            if new_event['type'] == 'oneway':
                event_desc += f" {new_event['reverse_edge'][0]}->{new_event['reverse_edge'][1]} removed"
            else:
                event_desc += f" {new_event['edge'][0]}->{new_event['edge'][1]}"
            log_events.append(event_desc)

        # Recomputiing paths for both cars if any events occurred and they're at a node
        if len(new_events) > 0:
            if carA['edge_from'] is None and carA['path']:
                _, path = Dijkstra(adj_, carA['pos'], exit_nodes)
                carA['path'] = path
                carA['path_index'] = 0

            if carB['delay'] == 0 and carB['edge_from'] is None and carB['path']:
                target_pos = carA['edge_to'] if carA['edge_to'] is not None else carA['pos']
                _, path = Dijkstra(adj_, carB['pos'], target_pos)
                carB['path'] = path
                carB['path_index'] = 0

        # Reverting back the edges after the events expired
        expired = []
        for event in active_events:
            if event['expires_at'] <= step:
                expired.append(event)

        for event in expired:
            revert_event(adj_, event)
            active_events.remove(event)

            # Recomputing the paths after the dynamic events time ends
            if carA['edge_from'] is None and carA['path']:
                _, path = Dijkstra(adj_, carA['pos'], exit_nodes)
                carA['path'] = path
                carA['path_index'] = 0

            if carB['delay'] == 0 and carB['edge_from'] is None and carB['path']:
                target_pos = carA['edge_to'] if carA['edge_to'] is not None else carA['pos']
                _, path = Dijkstra(adj_, carB['pos'], target_pos)
                carB['path'] = path
                carB['path_index'] = 0


        simulation_log.append({
            "step": step,
            "carA": {
                "pos": carA["pos"],
                "edge_from": carA.get("edge_from"),
                "edge_to": carA.get("edge_to"),
                "progress": carA.get("progress", 0),
                "Dijkstra_path": list(carA["path"]) if carA.get("path") else [],
            },
            "carB": {
                "pos": carB["pos"],
                "edge_from": carB.get("edge_from"),
                "edge_to": carB.get("edge_to"),
                "progress": carB.get("progress", 0),
                "Dijkstra_path": list(carB["path"]) if carB.get("path") else [],
            },
            "caught": caught,
            "reached": reached,
            "log_events": log_events
        })

        # Checking whether car A is caught or car A has escaped
        if caught:
            print(f"Car B caught Car A at node {carA['pos']}! Paris Police!! Saviour of the Louvre !!!")
            break

        elif reached:
            print("Car A reached exit first! The burglars vanish like thin air...")
            break

        elif step == m_steps-1:
            print("Both Car A and Car B are still in the city even after 50 steps! It's a long chase wohooo!!")


    return simulation_log, caught, reached



In [11]:
a,b,c,d = load_graph("graph_with_metadata.json")
x,y,z = chase(a,d,50)

Graph was successfully loaded in the code
Car B caught Car A at node 21! Paris Police!! Saviour of the Louvre !!!


**We check for the time taken for this chase**

In [12]:
len(x)

16

In [16]:
time_sum = 0
caught = 0
reached = 0

In [17]:
for i in range (1000):
  a,b,c,d = load_graph("graph_with_metadata.json")
  x,y,z = chase(a,d,50)

  time_sum += len(x)
  print(y)
  print(z)

  if(y): caught += 1
  if(z): reached += 1

Graph was successfully loaded in the code
Car A reached exit first! The burglars vanish like thin air...
False
True
Graph was successfully loaded in the code
Car A reached exit first! The burglars vanish like thin air...
False
True
Graph was successfully loaded in the code
Car A reached exit first! The burglars vanish like thin air...
False
True
Graph was successfully loaded in the code
Both Car A and Car B are still in the city even after 50 steps! It's a long chase wohooo!!
False
False
Graph was successfully loaded in the code
Car A reached exit first! The burglars vanish like thin air...
False
True
Graph was successfully loaded in the code
Car A reached exit first! The burglars vanish like thin air...
False
True
Graph was successfully loaded in the code
Both Car A and Car B are still in the city even after 50 steps! It's a long chase wohooo!!
False
False
Graph was successfully loaded in the code
Car A reached exit first! The burglars vanish like thin air...
False
True
Graph was succ

In [18]:
p_w_r = float(caught) / float(1000)     #police win rate
b_e_r = float(reached) / float(1000)    #burglar escape rate

a_c_d = float(time_sum) / float(1000)   #average chase duration

In [19]:
p_w_r

0.336

In [20]:
b_e_r

0.543

In [21]:
a_c_d

21.822